In [ ]:
"""
LST COMPARISON: NHDA vs. Reference Areas (RA)

Uses already created reference areas (RA).
Results are saved with yearly columns (averaged across tiles per year).
"""

import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point
import random
from datetime import datetime

# ============================================================================
# CONFIGURATION
# ============================================================================

base_path = Path(r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\processed_datasets\LST_Landsat")
output_path = Path(r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\LST_Comparison")

nhda_gpkg = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\New_Housing_Development_Areas\NHDA_with_construction_years_RF_AUC.gpkg"
ra_gpkg = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Reference_Areas\Reference_Areas.gpkg"

years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
n_random_points = 100

output_path.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("LST ANALYSIS: NHDA VS. REFERENCE AREAS (RA)")
print("=" * 80)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================================================
# 1. LOAD DATA
# ============================================================================

print("\n1. LOAD DATA")
print("=" * 80)

nhda_gdf = gpd.read_file(nhda_gpkg)

ra_gdf = gpd.read_file(ra_gpkg)

print(f"✓ NHDA areas: {len(nhda_gdf)}")
print(f"✓ Reference areas (RA): {len(ra_gdf)}")

if nhda_gdf.crs != ra_gdf.crs:
    ra_gdf = ra_gdf.to_crs(nhda_gdf.crs)

ra_dict = {row['nhda_id']: row.geometry for _, row in ra_gdf.iterrows()}

# ============================================================================
# 2. HELPER FUNCTIONS
# ============================================================================

def generate_random_points_in_geometry(geometry, n_points, max_attempts=None):
    """Generate n random points within a geometry."""
    if max_attempts is None:
        max_attempts = n_points * 100

    points = []
    attempts = 0
    minx, miny, maxx, maxy = geometry.bounds

    while len(points) < n_points and attempts < max_attempts:
        point = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if geometry.contains(point):
            points.append(point)
        attempts += 1

    return points


def extract_lst_values(points_gdf, raster_path):
    """Extract LST values for a GeoDataFrame of points from one raster."""
    try:
        with rasterio.open(raster_path) as src:
            points_reproj = points_gdf.to_crs(src.crs) if points_gdf.crs != src.crs else points_gdf

            values = []
            for _, point in points_reproj.iterrows():
                coords = [(point.geometry.x, point.geometry.y)]
                for val in src.sample(coords):
                    if src.nodata is not None and val[0] == src.nodata:
                        values.append(np.nan)
                    else:
                        values.append(val[0])

            return values
    except Exception:
        return [np.nan] * len(points_gdf)


# ============================================================================
# 3. FIND YEARLY LST IMAGES
# ============================================================================

print("\n2. FIND AVAILABLE YEARLY LST MEDIANS FOR BAVARIA")
print("=" * 80)

year_data_paths = {}

for year in years:
    lst_file = base_path / f"Landsat89_L2_LST_JJA_Median_Bayern_{year}.tif"

    if lst_file.exists():
        year_data_paths[year] = lst_file
        print(f"  {year}: found")
    else:
        print(f"  {year}: not found")

if not year_data_paths:
    raise FileNotFoundError("No yearly Bavaria LST median files found.")

# ============================================================================
# 4. MAIN LOOP
# ============================================================================

print("\n3. START ANALYSIS")
print("=" * 80)

all_results = []
all_geometries = []
all_points = []
failed_clusters = []

random.seed(42)

clusters_with_ra = [
    row for _, row in nhda_gdf.iterrows()
    if row['nhda_id'] in ra_dict
]

print(f"Analyzing {len(clusters_with_ra)} NHDA clusters with RA")

print("\nGenerating random points...")
cluster_points = {}

for i, nhda_row in enumerate(clusters_with_ra):
    if (i + 1) % 100 == 0:
        print(f"  {i + 1}/{len(clusters_with_ra)} clusters")

    nhda_id = nhda_row['nhda_id']
    nhda_geom = nhda_row.geometry
    ra_geom = ra_dict[nhda_id]

    nhda_points = generate_random_points_in_geometry(nhda_geom, n_random_points)
    ra_points = generate_random_points_in_geometry(ra_geom, n_random_points)

    if len(nhda_points) >= n_random_points * 0.5 and len(ra_points) >= n_random_points * 0.5:
        cluster_points[nhda_id] = {
            'nhda_points_gdf': gpd.GeoDataFrame(geometry=nhda_points, crs=nhda_gdf.crs),
            'ra_points_gdf': gpd.GeoDataFrame(geometry=ra_points, crs=nhda_gdf.crs),
            'nhda_geom': nhda_geom,
            'ra_geom': ra_geom,
            'nhda_area': nhda_geom.area,
            'ra_area': ra_geom.area
        }

        all_geometries.append({
            'geometry': nhda_geom,
            'nhda_id': nhda_id,
            'type': 'NHDA',
            'area_ha': nhda_geom.area / 10000
        })
        all_geometries.append({
            'geometry': ra_geom,
            'nhda_id': nhda_id,
            'type': 'RA',
            'area_ha': ra_geom.area / 10000
        })

        for j, point in enumerate(nhda_points):
            all_points.append({
                'geometry': point,
                'nhda_id': nhda_id,
                'type': 'NHDA',
                'point_id': j
            })

        for j, point in enumerate(ra_points):
            all_points.append({
                'geometry': point,
                'nhda_id': nhda_id,
                'type': 'RA',
                'point_id': j
            })
    else:
        failed_clusters.append(nhda_id)

print(f"✓ {len(cluster_points)} clusters with valid random points")

print("\nExtracting LST for all years...")

for year, lst_file in year_data_paths.items():
    print(f"\nYear {year}:")

    processed = 0

    for nhda_id, points_dict in cluster_points.items():
        try:
            nhda_values = extract_lst_values(
                points_dict['nhda_points_gdf'],
                lst_file
            )
            ra_values = extract_lst_values(
                points_dict['ra_points_gdf'],
                lst_file
            )

            nhda_values = [v for v in nhda_values if not np.isnan(v)]
            ra_values = [v for v in ra_values if not np.isnan(v)]

            if (
                len(nhda_values) >= n_random_points * 0.3
                and len(ra_values) >= n_random_points * 0.3
            ):
                nhda_median = np.median(nhda_values)
                ra_median = np.median(ra_values)
                difference = nhda_median - ra_median

                all_results.append({
                    'nhda_id': nhda_id,
                    'year': year,
                    'nhda_median_LST': nhda_median,
                    'ra_median_LST': ra_median,
                    'difference': difference,
                    'nhda_std': np.std(nhda_values),
                    'ra_std': np.std(ra_values),
                    'n_values_nhda': len(nhda_values),
                    'n_values_ra': len(ra_values),
                    'nhda_area_ha': points_dict['nhda_area'] / 10000,
                    'ra_area_ha': points_dict['ra_area'] / 10000
                })

                processed += 1

        except Exception:
            continue

    print(f"  ✓ {processed} clusters processed")

# ============================================================================
# 5. SAVE RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("SAVE RESULTS")
print("=" * 80)

if all_results:
    df_detailed = pd.DataFrame(all_results)
    output_csv_detailed = output_path / "LST_comparison_detailed_all_years.csv"
    df_detailed.to_csv(output_csv_detailed, index=False)
    print(f"✓ Detailed: {output_csv_detailed} ({len(df_detailed)} rows)")

    df_yearly = df_detailed.groupby('year').agg({
        'nhda_median_LST': 'median',
        'ra_median_LST': 'median',
        'difference': ['mean', 'median', 'std'],
        'nhda_id': 'count'
    }).reset_index()
    df_yearly.columns = ['year', 'nhda_median', 'ra_median', 'diff_mean', 'diff_median', 'diff_std', 'n_clusters']

    output_csv_yearly = output_path / "LST_comparison_yearly_aggregated.csv"
    df_yearly.to_csv(output_csv_yearly, index=False)
    print(f"✓ Yearly aggregated: {output_csv_yearly}")

    print("\nCreating year-based columns for geometries...")

    pivot_nhda = df_detailed.pivot(index='nhda_id', columns='year', values='nhda_median_LST')
    pivot_nhda.columns = [f'nhda_median_LST_{year}' for year in pivot_nhda.columns]

    pivot_ra = df_detailed.pivot(index='nhda_id', columns='year', values='ra_median_LST')
    pivot_ra.columns = [f'ra_median_LST_{year}' for year in pivot_ra.columns]

    pivot_diff = df_detailed.pivot(index='nhda_id', columns='year', values='difference')
    pivot_diff.columns = [f'difference_LST_{year}' for year in pivot_diff.columns]

    pivot_nhda_std = df_detailed.pivot(index='nhda_id', columns='year', values='nhda_std')
    pivot_nhda_std.columns = [f'nhda_std_LST_{year}' for year in pivot_nhda_std.columns]

    pivot_ra_std = df_detailed.pivot(index='nhda_id', columns='year', values='ra_std')
    pivot_ra_std.columns = [f'ra_std_LST_{year}' for year in pivot_ra_std.columns]

    df_cluster_yearly = pd.concat([pivot_nhda, pivot_ra, pivot_diff, pivot_nhda_std, pivot_ra_std], axis=1)
    df_cluster_yearly = df_cluster_yearly.reset_index()

    avg_by_cluster = df_detailed.groupby('nhda_id').agg(
        nhda_median_LST_avg=('nhda_median_LST', 'median'),
        ra_median_LST_avg=('ra_median_LST', 'median'),
        difference_LST_avg=('difference', 'median'),
        n_years=('year', 'count')
    ).reset_index()

    df_cluster_yearly = df_cluster_yearly.merge(avg_by_cluster, on='nhda_id', how='left')

    output_csv_cluster = output_path / "LST_comparison_cluster_yearly.csv"
    df_cluster_yearly.to_csv(output_csv_cluster, index=False)
    print(f"✓ Cluster yearly data: {output_csv_cluster} ({len(df_cluster_yearly)} clusters)")

    if all_geometries:
        gdf_geom = gpd.GeoDataFrame(all_geometries, crs=nhda_gdf.crs)
        gdf_geom = gdf_geom.merge(df_cluster_yearly, on='nhda_id', how='left')

        output_gpkg = output_path / "LST_analysis_geometries.gpkg"
        gdf_geom.to_file(output_gpkg, driver='GPKG', layer='analysis_areas')
        print(f"✓ Geometries: {output_gpkg}")

        lst_cols = [col for col in gdf_geom.columns if 'median_LST' in col or 'difference_LST' in col]
        print(f"  → Created {len(lst_cols)} LST columns")
        print(f"  → Example columns: {lst_cols[:5]}")

    if all_points:
        gdf_points = gpd.GeoDataFrame(all_points, crs=nhda_gdf.crs)
        output_gpkg_points = output_path / "LST_analysis_points.gpkg"
        gdf_points.to_file(output_gpkg_points, driver='GPKG', layer='random_points')
        print(f"✓ Points: {output_gpkg_points}")

    print("\n" + "=" * 80)
    print("STATISTICS")
    print("=" * 80)
    print(f"Analyzed clusters: {len(df_cluster_yearly)}")
    print(f"Failed: {len(failed_clusters)}")

    print("\nLST medians (across all years):")
    print(f"  NHDA: {df_cluster_yearly['nhda_median_LST_avg'].median():.2f}")
    print(f"  RA: {df_cluster_yearly['ra_median_LST_avg'].median():.2f}")
    print(f"  Difference (NHDA - RA): {df_cluster_yearly['difference_LST_avg'].median():.2f}")

    warmer = (df_cluster_yearly['difference_LST_avg'] > 0).sum()
    colder = (df_cluster_yearly['difference_LST_avg'] < 0).sum()
    print("\nDistribution:")
    print(f"  NHDA warmer than RA: {warmer} ({100 * warmer / len(df_cluster_yearly):.1f}%)")
    print(f"  NHDA colder than RA: {colder} ({100 * colder / len(df_cluster_yearly):.1f}%)")

    print("\nYearly development:")
    for _, row in df_yearly.iterrows():
        print(f"  {int(row['year'])}: Δ={row['diff_mean']:.2f} (Median={row['diff_median']:.2f}, n={int(row['n_clusters'])})")

    print("\n" + "=" * 80)
    print("CREATE VISUALIZATIONS")
    print("=" * 80)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(df_cluster_yearly['difference_LST_avg'], bins=30, edgecolor='black', alpha=0.7, color='orange')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No difference')
    ax.axvline(x=df_cluster_yearly['difference_LST_avg'].median(), color='blue', linestyle='--', linewidth=2, label='Median')
    ax.set_xlabel('LST Difference (NHDA - RA) [°C]', fontsize=12)
    ax.set_ylabel('Number of Clusters', fontsize=12)
    ax.set_title('Distribution of LST Differences (median over all years)', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    output_plot1 = output_path / "LST_difference_distribution.png"
    plt.savefig(output_plot1, dpi=300, bbox_inches='tight')
    print(f"✓ Histogram: {output_plot1}")
    plt.close()

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    ax1 = axes[0]
    ax1.plot(df_yearly['year'], df_yearly['nhda_median'],
             'o-', linewidth=2.5, markersize=9, label='NHDA', color='red')
    ax1.plot(df_yearly['year'], df_yearly['ra_median'],
             's-', linewidth=2.5, markersize=9, label='RA', color='green')
    ax1.set_ylabel('LST Median [°C]', fontsize=12, fontweight='bold')
    ax1.set_title('LST Development: NHDA vs. RA (Median)', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    ax2 = axes[1]
    ax2.plot(df_yearly['year'], df_yearly['diff_mean'],
             'o-', linewidth=2.5, markersize=9, color='blue')
    ax2.fill_between(df_yearly['year'],
                     df_yearly['diff_mean'] - df_yearly['diff_std'],
                     df_yearly['diff_mean'] + df_yearly['diff_std'],
                     alpha=0.2, color='blue')
    ax2.axhline(y=0, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax2.set_xlabel('Year', fontsize=12, fontweight='bold')
    ax2.set_ylabel('LST Difference (NHDA - RA) [°C]', fontsize=12, fontweight='bold')
    ax2.set_title('LST Difference over Time', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    output_plot2 = output_path / "LST_timeseries_comparison.png"
    plt.savefig(output_plot2, dpi=300, bbox_inches='tight')
    print(f"✓ Time series: {output_plot2}")
    plt.close()

    fig, ax = plt.subplots(figsize=(10, 10))

    scatter = ax.scatter(df_detailed['ra_median_LST'],
                         df_detailed['nhda_median_LST'],
                         c=df_detailed['year'],
                         alpha=0.5, s=30, cmap='viridis', edgecolors='black', linewidth=0.5)

    min_val = min(df_detailed['ra_median_LST'].min(), df_detailed['nhda_median_LST'].min())
    max_val = max(df_detailed['ra_median_LST'].max(), df_detailed['nhda_median_LST'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='1:1 Line')

    ax.set_xlabel('RA LST Median [°C]', fontsize=12)
    ax.set_ylabel('NHDA LST Median [°C]', fontsize=12)
    ax.set_title('LST Comparison: NHDA vs. RA (all years, Median)', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Year', fontsize=11)

    plt.tight_layout()
    output_plot3 = output_path / "LST_scatter_comparison.png"
    plt.savefig(output_plot3, dpi=300, bbox_inches='tight')
    print(f"✓ Scatter: {output_plot3}")
    plt.close()

else:
    print("⚠ No results!")

print("\n" + "=" * 80)
print("✅ DONE")
print("=" * 80)
print(f"End: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

LST ANALYSIS: NHDA VS. REFERENCE AREAS (RA)
Start: 2026-07-27 14:24:47

1. LOAD DATA
✓ NHDA areas: 839
✓ Reference areas (RA): 839

2. FIND AVAILABLE YEARLY LST MEDIANS FOR BAVARIA
  2015: found
  2016: found
  2017: found
  2018: found
  2019: found
  2020: found
  2021: found
  2022: found
  2023: found
  2024: found
  2025: found

3. START ANALYSIS
Analyzing 839 NHDA clusters with RA

Generating random points...
  100/839 clusters
  200/839 clusters
  300/839 clusters
  400/839 clusters
  500/839 clusters
  600/839 clusters
  700/839 clusters
  800/839 clusters
✓ 839 clusters with valid random points

Extracting LST for all years...

Year 2015:
  ✓ 800 clusters processed

Year 2016:
  ✓ 810 clusters processed

Year 2017:
  ✓ 809 clusters processed

Year 2018:
  ✓ 799 clusters processed

Year 2019:
  ✓ 812 clusters processed

Year 2020:
  ✓ 812 clusters processed

Year 2021:
  ✓ 809 clusters processed

Year 2022:
  ✓ 812 clusters processed

Year 2023:
  ✓ 812 clusters processed

Year